# STAGE C/5 — Fine-tune NER **v2** (KB-xương-sống + khử nhiễu)

**Add Input (BẮT BUỘC):** output **Stage A** (dapt) + **Stage B** (silver, llm) + **Stage D** (kb).
**Settings:** GPU **T4 (1 con)** + Internet ON. **~30–40′**. Xong → **Commit** cho Stage E.

v2: nhãn **KB biên sạch** (distant) + silver/llm **ĐÃ KHỬ NHIỄU** + template → hết học rác.


In [ ]:
# Clone repo
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: tránh DataParallel chậm
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
!pip install -q rapidfuzz pyyaml regex accelerate sentencepiece


In [ ]:
# Tìm artifact từ stage trước (đã Add Input)
import glob, os, shutil
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
def find_model(name):
    # model dir = thư mục có config.json (TRÁNH khớp nhầm src/ner là code)
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}/config.json", recursive=True)
            if "/src/" not in h]
    return os.path.dirname(hits[0]) if hits else None


In [ ]:
import glob, os, shutil
DAPT = find_model('dapt') or 'xlm-roberta-large'
SILVER = find_input('silver.jsonl') or ''
LLM = find_input('llm.jsonl') or ''
KB = find_input('kb')
if KB:                                   # nạp KB (Stage D) để distant + KB-trim
    for f in glob.glob(KB + '/*'):
        if os.path.isfile(f): shutil.copy(f, 'data/kb/' + os.path.basename(f))
    print('KB loaded:', os.listdir('data/kb'))
print('DAPT:', DAPT, '| SILVER:', bool(SILVER), '| LLM:', bool(LLM), '| KB:', bool(KB))
if DAPT == 'xlm-roberta-large': print('!! chua Add Input Stage A -> mat DAPT')
if not KB: print('!! chua Add Input Stage D (kb) -> distant/khu-nhieu yeu')


In [ ]:
# v2: sinh DISTANT (KB làm nhãn biên sạch) + KHỬ NHIỄU silver/llm
_a = (f' --silver {SILVER}' if SILVER else '') + (f' --llm {LLM}' if LLM else '')
!python scripts/prep_v2.py{_a} --out_dir data/v2


In [ ]:
# template (sàn, offset chuẩn)
!python scripts/gen_synth.py --n_train 4000 --n_dev 400


In [ ]:
# gộp nguồn v2 [distant + silver_clean + llm_clean + template] rồi fine-tune
import os
cands = ['data/v2/distant.jsonl','data/v2/silver_clean.jsonl','data/v2/llm_clean.jsonl','data/synthetic/train.jsonl']
parts = [p for p in cands if os.path.exists(p)]
train_arg = ','.join(parts)
print('TRAIN:', train_arg)
!python scripts/train_ner.py --model {DAPT} --train {train_arg} \
   --epochs 3 --batch_size 8 --grad_accum 2 --max_length 384 --optim adafactor \
   --out models/ner


In [ ]:
# Lưu model ra /kaggle/working để Commit
import shutil, os
dst = "/kaggle/working/ner"
if os.path.isdir("models/ner"):
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree("models/ner", dst)
    print("SAVED ->", dst, "| Commit roi qua Stage E")
else:
    print("!! train chưa tạo models/ner")


**OOM?** `--batch_size 4 --grad_accum 4`. **prep_v2 cần KB (Stage D)** cho distant + KB-trim; thiếu KB vẫn chạy nhưng yếu.
